# ⚔️ B1　Boss 戰：旅費精算師
**Python 冒險之旅 2026**　｜　Day 1（08/29 六）🏝️ 起始之島　｜　Boss 戰　｜　🏅 200 XP

📖 對應教科書：第 1–2 章綜合


### 🎯 這一關你會學到
- 綜合運用變數、運算子與格式化輸出完成小專案

### 🧭 闖關方式
1. 先按下方「🧰 魔法工具箱」那一格左邊的 ▶（第一次執行 Colab 會花幾秒鐘連線）。
2. 依序閱讀說明、執行範例、完成每個「🎯 任務」，再執行它下面的「檢查」格。
3. 看到 ✅ 就往下一個任務；看到 ❌ 就依提示修改，再重新執行任務格與檢查格。
4. 全部通過後，執行最下面的「🔑 通關密語」格，把密語貼回 [入口網頁](https://johnnychao.github.io/python-quest-2026/)。

> 💾 建議先點選「檔案 → 在雲端硬碟中儲存副本」，你的進度才會留在自己的 Google 雲端硬碟。

In [ ]:
#@title 🧰 魔法工具箱：先在右邊填「暱稱」，再按左邊的 ▶ 執行這一格 { display-mode: "form" }
暱稱 = "" #@param {type:"string"}
# ======================================================================
#  Python 冒險之旅 2026 · 關卡檢查工具（看不懂沒關係，這一格不是今天的功課 😉）
# ======================================================================
import hashlib, unicodedata, io, sys, re, contextlib, traceback, builtins

_LEVEL = "B1"
_SALT = "python-quest-2026-datama"
_TASKS = ["B1-1", "B1-2", "B1-3", "B1-4", "B1-5"]
_XP_EACH = 40
_CHECKS = {}
_PASSED = builtins.__dict__.setdefault("_pyquest_" + _LEVEL, {})
_HINTS = {}

def _norm_name(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s))).lower()

def _squash(s):
    return re.sub(r"\s+", "", str(s))

def 出現(out, *subs):
    """輸出中是否（忽略空白）包含所有片段"""
    o = _squash(out)
    return all(_squash(x) in o for x in subs)

def 數字們(out):
    """抓出輸出裡所有的數字（float）"""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(out))]

def 行列表(out):
    return [ln.rstrip() for ln in str(out).splitlines() if ln.strip()]

class _NeedMoreInput(Exception):
    pass

_HIST = builtins.__dict__.setdefault("_pyquest_hist", [])
def _on_pre_run(*args):
    try:
        info = args[0]
        src = getattr(info, "raw_cell", None)
        if isinstance(src, str):
            _HIST.append(src)
    except Exception:
        pass
try:
    _ip = get_ipython()
    if not builtins.__dict__.get("_pyquest_hooked"):
        _ip.events.register("pre_run_cell", _on_pre_run)
        builtins.__dict__["_pyquest_hooked"] = True
except Exception:
    pass

def _history():
    try:
        ip = get_ipython()
        h = list(ip.user_ns.get("In") or ip.user_ns.get("_ih") or [])
    except Exception:
        h = list(globals().get("In") or [])
    return [c for c in (h + list(_HIST)) if isinstance(c, str)]

def _find_cell(tid):
    marker = "# 🎯 任務 " + tid
    for cell in reversed(_history()):
        if marker in cell:
            lines = [ln for ln in cell.splitlines()
                     if not re.match(r"\s*(檢查|通關密語)\s*\(", ln)]
            return "\n".join(lines)
    return None

def _make_runner(src):
    def run(*inputs):
        feed = iter([str(x) for x in inputs])
        buf = io.StringIO()
        try:
            ns = dict(get_ipython().user_ns)
        except Exception:
            ns = dict(globals())
        def _fake_input(prompt=""):
            try:
                return next(feed)
            except StopIteration:
                raise _NeedMoreInput()
        ns["input"] = _fake_input
        ns["__name__"] = "__main__"
        try:
            import matplotlib.pyplot as _plt
            _plt.close("all"); _orig_show = _plt.show; _plt.show = lambda *a, **k: None
        except Exception:
            _plt = None
        try:
            with contextlib.redirect_stdout(buf):
                exec(compile(src, "<任務 " + _LEVEL + ">", "exec"), ns)
        finally:
            if _plt is not None:
                _plt.show = _orig_show
        return buf.getvalue(), ns
    run.src = src
    return run

def 任務定義(tid, fn, 提示=""):
    _CHECKS[tid] = fn
    _HINTS[tid] = 提示

def _progress():
    done = sum(1 for t in _TASKS if _PASSED.get(t))
    bar = "■" * done + "□" * (len(_TASKS) - done)
    return f"[{bar}] {done}/{len(_TASKS)}"

def 檢查(tid):
    tid = str(tid)
    if tid not in _CHECKS:
        print(f"⚠️ 找不到任務 {tid} 的檢查設定。"); return
    src = _find_cell(tid)
    if src is None:
        print(f"❌ 找不到「# 🎯 任務 {tid}」的程式格。請先執行那一格（並保留第一行的標記），再執行這裡。")
        return
    run = _make_runner(src)
    try:
        result = _CHECKS[tid](run)
    except _NeedMoreInput:
        result = (False, "你的程式呼叫 input() 的次數比題目預期的多，請檢查輸入的次數。")
    except Exception as e:
        tb = traceback.format_exc().strip().splitlines()[-1]
        result = (False, f"程式執行時發生錯誤 → {tb}")
    ok, extra = (result, "") if isinstance(result, bool) else result
    if ok:
        first = not _PASSED.get(tid)
        _PASSED[tid] = True
        print(f"✅ 任務 {tid} 通過！{'+' + str(_XP_EACH) + ' XP ' if first else ''}{_progress()}")
        if all(_PASSED.get(t) for t in _TASKS):
            print("🏆 本關所有任務都完成了！請執行最下面的「通關密語」那一格。")
    else:
        print(f"❌ 任務 {tid} 還沒通過。{_progress()}")
        if extra: print("   💬 " + str(extra))
        if _HINTS.get(tid): print("   💡 提示：" + _HINTS[tid])
        print("   👉 修改程式後，先重新執行任務那一格，再執行這一格。")

def 通關密語():
    missing = [t for t in _TASKS if not _PASSED.get(t)]
    if missing:
        print("🔒 還有任務未通過：" + "、".join(missing) + "　完成後再來拿密語吧！")
        return
    name = 暱稱.strip() if isinstance(暱稱, str) else ""
    if not name:
        name = input("請輸入你在入口網頁登錄的暱稱：").strip()
    if not name:
        print("⚠️ 暱稱不能是空白。"); return
    code = hashlib.sha256(f"{_SALT}|{_LEVEL}|{_norm_name(name)}".encode("utf-8")).hexdigest()[:6].upper()
    print("=" * 46)
    print(f"🎉 恭喜 {name}！{_LEVEL} 通關！")
    print(f"🔑 通關密語：PYQ-{_LEVEL}-{code}")
    print("👉 回到入口網頁，把密語貼到這一關的「輸入通關密語」欄位。")
    print("=" * 46)

print(f"🧰 魔法工具箱已準備好！本關有 {len(_TASKS)} 個任務：{'、'.join(_TASKS)}")
print("   做完每個任務後，執行它下方的「檢查」格；全部通過後執行最下方的「通關密語」。")

# ---------------- 各任務的檢查規則 ----------------
def _check_B1_1(run):
    out, ns = run()
    return (ns.get("total") == 26350, f"total 應該是 26350，現在是 {ns.get('total')}。")
任務定義("B1-1", _check_B1_1, 提示="住宿是 hotel * (days - 1)，餐費是 food * days。")

def _check_B1_2(run):
    out, ns = run()
    return ((ns.get("each"), ns.get("left")) == (8783, 1), "each 應該是 8783、left 是 1。")
任務定義("B1-2", _check_B1_2, 提示="each = fund // people；left = fund % people。")

def _check_B1_3(run):
    out, ns = run()
    if abs(float(ns.get("jpy", 0)) - 121210) > 1e-6: return (False, "jpy 應該是 121210.0。")
    return ("121,210" in out, "要用 {jpy:,.0f} 顯示千分位且不顯示小數。")
任務定義("B1-3", _check_B1_3, 提示="格式 {jpy:,.0f}：逗號是千分位，.0f 是 0 位小數。")

def _check_B1_4(run):
    out, ns = run()
    if ns.get("ok") is not False: return (False, "26350 > 25000，ok 應該是 False（要用比較運算子算出來）。")
    return (ns.get("over") == 1350, "over 應該是 1350。")
任務定義("B1-4", _check_B1_4, 提示="ok = total <= budget；over = total - budget。")

def _check_B1_5(run):
    out, ns = run()
    if not 出現(out, "12,000", "9,600", "4,750", "26,350"): return (False, "四個金額都要有千分位。")
    lines = 行列表(out)
    if out.count("=" * 14) < 2: return (False, "要有兩條 14 個 = 的分隔線。")
    return (lines[-1].startswith("總計") and lines[-1].endswith("26,350"), "最後一行應該是 總計 ... 26,350。")
任務定義("B1-5", _check_B1_5, 提示="每一行的寫法都和「機票」那一行一樣，只是換變數。")


## ⚔️ Boss 登場：旅費精算師
勇者小隊要去日本自助旅行 5 天。你是隊上的精算師，請用今天學的**變數、運算子、格式化輸出**完成旅費試算。

| 項目 | 數值 |
|---|---|
| 天數 `days` | 5 |
| 機票 `ticket` | 12000 元（每人） |
| 住宿 `hotel` | 每晚 2400 元（住 days − 1 晚） |
| 餐費 `food` | 每天 950 元 |
| 匯率 `rate` | 1 台幣 = 4.6 日圓 |
| 預算 `budget` | 25000 元 |
| 人數 `people` | 3 人 |

> Boss 戰沒有新語法，只有「把學過的東西組合起來」。卡住時回頭看 L02、L03 的範例。

In [ ]:
# 先執行這一格，建立基本資料（之後的任務都會用到）
days, ticket, hotel, food = 5, 12000, 2400, 950
rate, budget, people = 4.6, 25000, 3
print("資料準備完成")

### 🎯 任務 B1-1　每人總花費

計算一個人的總花費 `total` = 機票 + 住宿（days−1 晚）+ 餐費（days 天），並印出 `每人總花費：26350 元`。

In [ ]:
# 🎯 任務 B1-1　每人總花費（請保留這一行）
days, ticket, hotel, food = 5, 12000, 2400, 950
total = ???
print("每人總花費：", total, "元")

In [ ]:
檢查("B1-1")   # ◀ 執行這一格，看看任務 B1-1 有沒有過關

### 🎯 任務 B1-2　公費平分

隊上的公費 `fund = 26350` 元要平分給 3 人。用 `//` 算出每人拿多少（`each`），用 `%` 算出剩多少（`left`），印出 `每人 8783 元，剩 1 元`。

In [ ]:
# 🎯 任務 B1-2　公費平分（請保留這一行）
fund, people = 26350, 3
each = ???
left = ???
print(f"每人 {each} 元，剩 {left} 元")

In [ ]:
檢查("B1-2")   # ◀ 執行這一格，看看任務 B1-2 有沒有過關

### 🎯 任務 B1-3　換成日圓

把 `total = 26350` 元換成日圓（`rate = 4.6`），存到 `jpy`，並用千分位、**不顯示小數**印出：`約 121,210 日圓`。

In [ ]:
# 🎯 任務 B1-3　換成日圓（請保留這一行）
total, rate = 26350, 4.6
jpy = ???
print(f"約 {jpy:???} 日圓")

In [ ]:
檢查("B1-3")   # ◀ 執行這一格，看看任務 B1-3 有沒有過關

### 🎯 任務 B1-4　預算夠不夠？

用**關係運算子**判斷：`ok` = 總花費是否不超過預算；`over` = 超出多少（可能是負數）。印出兩個值。（今天還沒學 if，用運算式就能回答！）

In [ ]:
# 🎯 任務 B1-4　預算夠不夠？（請保留這一行）
total, budget = 26350, 25000
ok = ???
over = ???
print("預算足夠？", ok)
print("超出預算：", over, "元")

In [ ]:
檢查("B1-4")   # ◀ 執行這一格，看看任務 B1-4 有沒有過關

### 🎯 任務 B1-5　精算報表

印出對齊的報表（項目靠左佔 6 格、金額靠右佔 8 格並加千分位）。最後一行總計要用 `=` 分隔線。

```
項目        金額
==============
機票      12,000
住宿       9,600
餐費       4,750
==============
總計      26,350
```

In [ ]:
# 🎯 任務 B1-5　精算報表（請保留這一行）
ticket, hotel_total, food_total = 12000, 9600, 4750
total = ticket + hotel_total + food_total
print(f"{'項目':<6}{'金額':>6}")
print("=" * 14)
print(f"{'機票':<6}{ticket:>8,}")
# 印出住宿、餐費、分隔線與總計

In [ ]:
檢查("B1-5")   # ◀ 執行這一格，看看任務 B1-5 有沒有過關

## 🌟 進階挑戰（不計分，給跑得快的勇者）
1. 把匯率改成 4.8，重新試算；想想哪些變數會跟著改變？
2. 加上「紀念品預算」變數，每人 3000 元，重新計算總花費與是否超出預算。
3. 用 `print()` 的 `end=''` 參數，把報表的「總計」印在同一行。

---
## 🔑 通關密語

全部任務都 ✅ 之後，執行下面這一格，會得到你專屬的通關密語（和暱稱綁定，每個人不一樣）。

In [ ]:
通關密語()

---
### 🧭 接下來
**下一關：🔤 L04 字串與格式化輸出入** → [在 Colab 開啟](https://colab.research.google.com/github/johnnychao/python-quest-2026/blob/main/notebooks/L04_strings_io.ipynb)

回到入口網頁：https://johnnychao.github.io/python-quest-2026/